# SoftmaxShield: QLoRA Fine-Tuning for Hallucination Prevention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook provides a complete, Google Colab-compatible instruction fine-tuning pipeline using **QLoRA (4-bit Quantized Low-Rank Adaptation)** on the SoftmaxShield dataset.

### Target Architecture & Model Selection
- **Base Model**: `Qwen/Qwen2.5-Coder-1.5B-Instruct` (1.54B parameters)
- **Alternative Compatible Models**: `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`, `bigcode/starcoder2-3b`
- **Target Hardware**: NVIDIA Tesla T4 GPU (Google Colab Free Tier / Kaggle)

### T4 GPU Memory Breakdown (15.0 GB VRAM Budget):
| Component | Precision | VRAM Consumption | Notes |
|---|---|---|---|
| **Base Model Weights** | 4-bit NormalFloat (NF4) | ~1.18 GB | $1.54 \times 10^9 \times 0.5$ bytes + double quant |
| **LoRA Trainable Adapters** | 16-bit Float (FP16) | ~0.04 GB | $r=16, \alpha=32$ across 7 projection matrices (~18M params) |
| **Optimizer States** | Paged AdamW 8-bit | ~0.15 GB | Pages out to CPU RAM under memory spikes |
| **Activation Memory** | FP16 with Checkpointing | ~2.20 GB | Batch size 4, sequence length 512 |
| **CUDA Context & Overhead** | - | ~0.45 GB | Driver and runtime overhead |
| **Total Peak VRAM** | - | **~4.02 GB** | **Leaves ~10.98 GB headroom (Zero OOM Risk)** |

## 1. Environment Setup & Dependency Installation
Install Hugging Face `transformers`, `peft`, `trl`, `bitsandbytes`, `datasets`, and `accelerate`.

In [ ]:
!pip install -q -U torch transformers datasets peft trl bitsandbytes accelerate

## 2. Hardware Verification

In [ ]:
import torch

assert torch.cuda.is_available(), "CUDA GPU is required. In Colab: Runtime -> Change runtime type -> T4 GPU."
print(f"Device Name     : {torch.cuda.get_device_name(0)}")
print(f"Total VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
!nvidia-smi

## 3. Load Base Model in 4-bit NormalFloat (NF4)
- `bnb_4bit_quant_type="nf4"`: Quantile quantization designed for normally distributed neural weights.
- `bnb_4bit_use_double_quant=True`: Second quantization step on quantization constants.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

model_id = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model = prepare_model_for_kbit_training(base_model)

vram_used = torch.cuda.memory_allocated() / 1e9
print(f"Base model loaded into VRAM: {vram_used:.2f} GB")

## 4. Base Model Baseline Inference (Before Fine-Tuning)
Demonstrates how the unmodified base model responds to a prompt requesting a removed API (`pandas.DataFrame.append()`).

In [ ]:
test_prompt = (
    "<|im_start|>system\n"
    "You are a Python programming assistant.<|im_end|>\n"
    "<|im_start|>user\n"
    "Use pandas.DataFrame.append() to add a row {'val': 42} to df in Pandas 2.2.<|im_end|>\n"
    "<|im_start|>assistant\n"
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    baseline_outputs = base_model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
baseline_response = tokenizer.decode(baseline_outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== BASELINE MODEL OUTPUT (Prone to Sycophantic Hallucination) ===")
print(baseline_response)

## 5. Configure LoRA Adapters
- Attaches low-rank decomposition matrices $W = W_0 + B \times A$ to all query, key, value, output, and MLP projection layers.

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

## 6. Ingest and Format the SoftmaxShield Dataset
Splits 2,100 records into 90% train (1,890) and 10% eval (210).

In [ ]:
from datasets import load_dataset

# In Colab: Upload data/finetuning_dataset.jsonl or fetch via curl/git
dataset = load_dataset("json", data_files="data/finetuning_dataset.jsonl", split="train")
splits = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Training set samples  : {len(train_ds)}")
print(f"Validation set samples: {len(eval_ds)}")

def format_sample(example):
    prompt_text = example["prompt"]
    lib_ctx = f"[Library: {example['library']}=={example['version']}]"
    if example.get("negative_example"):
        output_text = f"DISALLOWED: {example['explanation']}\n\n```python\n{example['correct_code']}\n```"
    else:
        output_text = f"```python\n{example['correct_code']}\n```"

    return (
        f"<|im_start|>system\n"
        f"You are a verified Python code generation assistant adhering to SoftmaxShield ground-truth APIs.<|im_end|>\n"
        f"<|im_start|>user\n"
        f"{prompt_text}\n{lib_ctx}<|im_end|>\n"
        f"<|im_start|>assistant\n"
        f"{output_text}<|im_end|>"
    )

## 7. Training Execution with SFTTrainer & Metric Tracking

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

output_dir = "./softmaxshield_adapter_checkpoints"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_steps=100,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=50,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_config,
    formatting_func=format_sample,
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

print("Starting QLoRA training on T4 GPU...")
trainer.train()

## 8. Quantitative Evaluation & Adapter Export

In [ ]:
# Compute evaluation loss on held-out validation set
eval_results = trainer.evaluate()
print("Evaluation Metrics:", eval_results)

# Export trained LoRA adapter
export_path = "./softmaxshield_qlora_adapter"
trainer.model.save_pretrained(export_path)
tokenizer.save_pretrained(export_path)
print(f"LoRA adapter weights saved successfully to {export_path}!")

## 9. Inference Comparison: Post-Fine-Tuning Verification
Test the fine-tuned adapter on the identical prompt that caused the baseline model to hallucinate.

In [ ]:
model.eval()
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    fine_tuned_outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

ft_response = tokenizer.decode(fine_tuned_outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== FINE-TUNED MODEL OUTPUT (Grounded Refusal & Modern Correction) ===")
print(ft_response)

# Note: The fine-tuned adapter successfully recognizes that DataFrame.append() is disallowed in Pandas 2.2
# and generates the verified alternative using pandas.concat([df, new_row]).